# HW1 — Introduction & Financial Data

2026 cohort · [Official assignment](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework1.md) · [Submission](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01)

This is an **unexecuted reproducibility notebook**, written from the 2026 homework instructions. It is **not** a transcript of the user's past submission; run the cells in order and inspect printed results before submitting. Source websites and Yahoo Finance are time-dependent. No generated results are claimed to be verified.

In [ ]:
%pip -q install pandas numpy requests lxml yfinance

In [ ]:
import io, json, re
from datetime import date
import numpy as np
import pandas as pd
import requests
import yfinance as yf

HEADERS = {"User-Agent": "Mozilla/5.0"}
def download_close(symbol, start, end):
    d = yf.download(symbol, start=start, end=end, auto_adjust=False, progress=False, threads=False)
    if d.empty:
        raise RuntimeError(f"Empty data for {symbol}")
    c = d["Close"]
    return c.iloc[:, 0].dropna() if isinstance(c, pd.DataFrame) else c.dropna()

answers = {}
print("Pandas:", pd.__version__, "yfinance:", yf.__version__)

## Q1 — Current S&P 500 constituents: year added
Only **completed calendar years** from 2020 onward are considered. Wikipedia describes *current* members, not a full history of all additions and removals.

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers=HEADERS, timeout=30)
html.raise_for_status()
sp500 = pd.read_html(io.StringIO(html.text))[0]
date_col = next(c for c in sp500 if "date added" in str(c).lower())
added = pd.to_datetime(sp500[date_col], errors="coerce")
years = added.dt.year
counts = years[(years >= 2020) & (years < date.today().year)].value_counts().sort_values(ascending=False)
display(counts)
answers["q1"] = int(counts.idxmax()) if not counts.empty else None
print("Q1:", answers["q1"])

## Q2 — YTD comparison on 21 August 2026
The Yahoo Finance `end` argument is **exclusive**, so use 22 August to include the closing observation on 21 August (or the last earlier trading day). Start from the first available observation on/after 1 January.

In [ ]:
tickers = {
    "US": "^GSPC", "China": "000001.SS", "Hong Kong": "^HSI",
    "Australia": "^AXJO", "India": "^NSEI", "Canada": "^GSPTSE",
    "Germany": "^GDAXI", "UK": "^FTSE", "Japan": "^N225",
    "Mexico": "^MXX", "Brazil": "^BVSP"
}
ytd = {}
for country, ticker in tickers.items():
    try:
        c = download_close(ticker, "2026-01-01", "2026-08-22")
        ytd[country] = float(c.iloc[-1] / c.iloc[0] - 1)
    except Exception as ex:
        print("MISSING:", country, ticker, ex)
display(pd.Series(ytd).sort_values(ascending=False).rename("YTD growth"))
assert len(ytd) == len(tickers), "Missing index, cannot finalize Q2"
answers["q2"] = sum(v > ytd["US"] for k, v in ytd.items() if k != "US")
print("Q2:", answers["q2"])

## Q3 — Significant corrections between consecutive all-time highs
A correction is a drawdown of at least 5% from an ATH close. The homework explicitly asks for **consecutive ATH pairs**; unresolved drawdown after the last ATH is excluded.

In [ ]:
close = download_close("^GSPC", "1950-01-01", (date.today() + pd.Timedelta(days=1)).isoformat())
close = close.sort_index()
prior_high = close.cummax().shift(1).fillna(-np.inf)
ath_idx = np.flatnonzero((close > prior_high).to_numpy())
corrections = []
for i, j in zip(ath_idx[:-1], ath_idx[1:]):
    peak = float(close.iloc[i])
    trough = float(close.iloc[i:j+1].min())
    pct = (peak - trough) / peak * 100
    if pct >= 5:
        corrections.append({"start": str(close.index[i].date()),
                            "end": str(close.index[j].date()),
                            "drawdown_pct": pct,
                            "duration_days": (close.index[j]-close.index[i]).days})
corrections_df = pd.DataFrame(corrections)
if corrections_df.empty:
    raise RuntimeError("No correction events found; review source and calculation")
display(corrections_df[["drawdown_pct", "duration_days"]].quantile([.25,.5,.75]))
answers["q3_median_drawdown_pct"] = float(corrections_df["drawdown_pct"].median())
print("Q3 median drawdown (%):", answers["q3_median_drawdown_pct"])

## Q4 — AMZN positive earnings surprise and two-day stock return
Map announcement to Day 2 and calculate Day 1 → Day 3 closing return. Availability/number of earnings rows returned by Yahoo Finance may change.

In [ ]:
earn = yf.Ticker("AMZN").get_earnings_dates(limit=40)
assert earn is not None and not earn.empty, "Earnings dates unavailable"
sur_col = next(c for c in earn.columns if "surprise" in str(c).lower())
earn = earn.copy()
earn["trading_date"] = pd.to_datetime(earn.index, utc=True).date
earn["surprise"] = pd.to_numeric(earn[sur_col], errors="coerce")
prices = download_close("AMZN", "2019-01-01", (date.today() + pd.Timedelta(days=1)).isoformat())
daily = pd.DataFrame({"date": prices.index.date,
                      "return_2d": (prices.shift(-1) / prices.shift(1) - 1).to_numpy()})
events = earn[earn["surprise"] > 0].merge(daily, left_on="trading_date", right_on="date")
events = events.dropna(subset=["return_2d"])
assert not events.empty, "No aligned positive-surprise observations"
answers["q4_median_2d_return_pct"] = float(events["return_2d"].median() * 100)
answers["q4_surprise_return_corr"] = float(events["surprise"].corr(events["return_2d"]))
print("Q4 median two-day return (%):", answers["q4_median_2d_return_pct"])
print("Surprise/return correlation:", answers["q4_surprise_return_corr"])
display(events[["trading_date","surprise","return_2d"]].tail())

## Summary
Review computation dates, scaling (% vs ratio), and actual course form choices. These are exploratory market-data results, not immutable answers.

In [ ]:
print(json.dumps({"run_date": date.today().isoformat(), "answers": answers}, indent=2, default=str))